# NLP Day 1 · 실습 3. Bigram으로 다음 단어 예측하기

**연결 이론:** 이론 4 — 언어모델, N-gram, 조건부확률, 미관찰 단어 조합

## 1. 이번 실습에서 만들 것

말뭉치에서 연속된 두 단어(Bigram)의 **등장 횟수 → 조건부확률 → 다음 단어 예측** 과정을 구현합니다.

- **A형 함께 구현하기:** `나는 사과를 먹는다` 2회와 `나는 바나나를 먹는다` 1회 → `2/3`, `1/3`
- **B형 직접 완성하기:** 바나나 문장 2회 추가 → `next_word_probs()` 완성 → `0.4`, `0.6` 비교

표준 라이브러리만 사용하며, 복잡한 모델 학습 없이 빈도와 확률의 관계를 확인합니다.

## 2. A형 · 함께 구현하기

### 2.1 말뭉치에서 Bigram 만들기

Bigram은 문장에 **연속해서 등장하는 두 단어**를 하나의 쌍으로 묶습니다.

In [ ]:
from collections import Counter

corpus_a = [
    "나는 사과를 먹는다",
    "나는 사과를 먹는다",
    "나는 바나나를 먹는다",
]


def make_bigram_counts(corpus):
    pairs = []
    for sentence in corpus:
        tokens = sentence.split()
        for i in range(len(tokens) - 1):
            pairs.append((tokens[i], tokens[i + 1]))
    return Counter(pairs)


bigram_counts_a = make_bigram_counts(corpus_a)
for (prev_word, next_word), count in bigram_counts_a.items():
    print(f"{prev_word} → {next_word}: {count}회")

assert bigram_counts_a[("나는", "사과를")] == 2
assert bigram_counts_a[("나는", "바나나를")] == 1

### 2.2 같은 앞 단어 뒤에 나온 횟수로 확률 계산하기

문맥을 `나는`으로 고정하면 다음 단어는 `사과를`(2회), `바나나를`(1회)로 관찰되었습니다.

\[
P(\text{다음 단어}\mid\text{앞 단어})=
\frac{\text{해당 Bigram 등장 횟수}}{\text{같은 앞 단어를 가진 Bigram의 전체 등장 횟수}}
\]

In [ ]:
context = "나는"
matching_a = {
    word: count
    for (prev, word), count in bigram_counts_a.items()
    if prev == context
}
total_a = sum(matching_a.values())
probs_a = {word: count / total_a for word, count in matching_a.items()}

print("문맥:", context)
print("전체 등장 횟수:", total_a)
for word, probability in probs_a.items():
    print(f"P({word} | {context}) = {probability:.3f}")

pred_a = max(probs_a, key=probs_a.get)
print("가장 가능성이 높은 다음 단어:", pred_a)

assert total_a == 3
assert abs(probs_a["사과를"] - 2 / 3) < 1e-12
assert abs(probs_a["바나나를"] - 1 / 3) < 1e-12
assert pred_a == "사과를"

### 2.3 관찰되지 않은 단어 조합 확인하기

'나는 → 먹는다'가 말뭉치에 없으므로, 등장 횟수만 사용하면 해당 조건부확률은 0입니다. 다음 코드로 확인합니다.

In [ ]:
unseen_count = bigram_counts_a[("나는", "먹는다")]
print("'나는 → 먹는다' 등장 횟수:", unseen_count)
print("기존 빈도 기반 확률:", unseen_count / total_a)
assert unseen_count == 0

## 3. B형 · 직접 완성하기

### 3.1 응용 문제: 문장 추가 후 확률 함수 완성하기

기존 말뭉치에 `나는 바나나를 먹는다`를 **두 번 더 추가**합니다.

**과제:** `next_word_probs(context, counts)`에서 같은 앞 단어를 가진 Bigram의 등장 횟수를 합하고, 각 후보의 등장 횟수를 그 합으로 나누어 **확률 사전(dict)**을 반환하도록 `TODO`를 완성하세요.

토큰화와 Bigram 횟수 계산 코드는 제공됩니다. **핵심 구현은 분모 계산과 확률 변환**입니다.

In [ ]:
corpus_b = corpus_a + [
    "나는 바나나를 먹는다",
    "나는 바나나를 먹는다",
]
bigram_counts_b = make_bigram_counts(corpus_b)


def next_word_probs(context, counts):
    matching = {
        next_word: count
        for (prev_word, next_word), count in counts.items()
        if prev_word == context
    }
    # TODO 1: matching에 포함된 등장 횟수를 모두 합하세요.
    # TODO 2: 단어별 횟수를 전체 횟수로 나눈 확률 사전을 반환하세요.
    # 참고: 해당 문맥이 말뭉치에 없다면 빈 사전 {}을 반환해도 됩니다.
    return {}


probs_b = next_word_probs("나는", bigram_counts_b)
print("B형 계산 결과:", probs_b)

### 3.2 결과 검증과 해석

In [ ]:
if not probs_b:
    print("B형 TODO가 아직 완성되지 않았습니다. next_word_probs()를 수정하고 다시 실행하세요.")
else:
    assert set(probs_b) == {"사과를", "바나나를"}
    assert abs(probs_b["사과를"] - 0.4) < 1e-12
    assert abs(probs_b["바나나를"] - 0.6) < 1e-12
    assert abs(sum(probs_b.values()) - 1.0) < 1e-12
    pred_b = max(probs_b, key=probs_b.get)
    assert pred_b == "바나나를"

    print("사과를 확률:", f"{probs_b['사과를']:.2f}")
    print("바나나를 확률:", f"{probs_b['바나나를']:.2f}")
    print("확률 합:", f"{sum(probs_b.values()):.2f}")
    print("예측 결과:", pred_b)
    print("검사 통과: 문장이 추가되면서 예측 결과가 바뀌었습니다.")

## 4. 결과 확인과 이번 실습 간단하게 요약

**완료 기준:** B형에서 `P(사과를|나는)=2/5=0.4`, `P(바나나를|나는)=3/5=0.6`, 확률 합 `1`, 예측 결과 `바나나를`을 확인합니다.

**첫째,** Bigram은 연속된 두 단어의 등장 횟수를 기록합니다.

**둘째,** 조건부확률은 같은 앞 단어 뒤에 등장한 **후보 횟수 / 전체 횟수**로 계산합니다.

**셋째,** 말뭉치가 달라지면 등장 횟수와 예측 확률도 달라집니다.

**넷째,** 관찰하지 못한 단어 조합은 빈도만 이용하면 확률이 0이 되는 한계가 있습니다.

다음에는 단어를 학습 가능한 벡터로 나타내고, 단어 사이의 관계를 표현하는 **임베딩·Word2Vec**으로 확장합니다.